**Generated By Gemini, Prompted and Edited By Manim Community Nepal**

In [5]:
from manim import *

class ScalingAndShearing(Scene):
    def construct(self):
        # --- CONFIGURATION ---
        # Colors
        COLOR_X = RED_C   # Corresponds to i-hat and vertical grid lines
        COLOR_Y = GREEN_C # Corresponds to j-hat and horizontal grid lines
        COLOR_VEC = YELLOW
        COLOR_GRID_BG = GREY
        
        # --- SETUP LAYOUT ---
        
        # 1. Background Grid (The "Old" Grid - Static Reference)
        # This stays fixed to show the original coordinate space with numbers
        background_plane = NumberPlane(
            x_range=[-5, 5, 1],
            y_range=[-4, 4, 1],
            background_line_style={
                "stroke_color": COLOR_GRID_BG,
                "stroke_width": 1,
                "stroke_opacity": 0.3
            },
            axis_config={"color": COLOR_GRID_BG, "include_numbers": True}
        )
        background_plane.set_width(7)
        background_plane.to_edge(RIGHT, buff=0.5)

        # Get the origin for correct positioning (Moved up to use for moving_plane alignment)
        origin_point = background_plane.coords_to_point(0, 0)
        
        # 2. Moving Grid (The "New" Grid - Transforms)
        # This grid will warp. We color its lines to match the axes.
        moving_plane = NumberPlane(
            x_range=[-5, 5, 1],
            y_range=[-4, 4, 1],
            background_line_style={
                "stroke_width": 2,
                "stroke_opacity": 0.6
            },
            axis_config={
                "color": WHITE, 
                "include_numbers": False, # Numbers only on background to avoid clutter
                "stroke_opacity": 0.5
            }
        )
        # FIX: Align the center of moving_plane (which is 0,0) to the specific origin point of background_plane
        # instead of aligning to background_plane's bounding box (which is shifted by text labels).
        moving_plane.set_width(7).move_to(origin_point)
        
        # COLORING THE GRID LINES
        # Vertical lines (x=const) represent X-spacing -> Color them Red (like i-hat)
        moving_plane.background_lines[0].set_color(COLOR_X)
        # Horizontal lines (y=const) represent Y-spacing -> Color them Green (like j-hat)
        moving_plane.background_lines[1].set_color(COLOR_Y)

        # 3. Vectors on the Grid
        # Basis Vectors & Labels
        i_hat = Arrow(
            start=origin_point, 
            end=moving_plane.coords_to_point(1, 0), 
            color=COLOR_X, 
            buff=0, 
            stroke_width=6
        )
        i_label = MathTex("\\hat{i}", color=COLOR_X).next_to(i_hat.get_end(), DOWN, buff=0.1)
        # Group label with vector so it transforms
        i_group = VGroup(i_hat, i_label)

        j_hat = Arrow(
            start=origin_point, 
            end=moving_plane.coords_to_point(0, 1), 
            color=COLOR_Y, 
            buff=0, 
            stroke_width=6
        )
        j_label = MathTex("\\hat{j}", color=COLOR_Y).next_to(j_hat.get_end(), LEFT, buff=0.1)
        j_group = VGroup(j_hat, j_label)
        
        # Sample generic vector v
        vector = Arrow(
            start=origin_point, 
            end=moving_plane.coords_to_point(1, 1), 
            color=COLOR_VEC, 
            buff=0, 
            stroke_width=6
        )
        vector_label = MathTex("\\vec{v}", color=COLOR_VEC).next_to(vector.get_end(), UR, buff=0.1)
        vec_group = VGroup(vector, vector_label)
        
        # MAIN TRANSFORMATION GROUP
        # Everything that moves goes here
        grid_group = VGroup(moving_plane, i_group, j_group, vec_group)


        # --- HELPER FOR TEXT LAYOUT ---
        def create_sidebar(title_text, matrix_mobj, desc_lines):
            title = Text(title_text, font_size=32).to_edge(LEFT, buff=0.5).to_edge(UP, buff=0.5)
            
            matrix_mobj.next_to(title, DOWN, buff=0.5).align_to(title, LEFT)
            
            descriptions = VGroup()
            
            kw_colors = {
                "$x$": COLOR_X,
                "$y$": COLOR_Y,
                "$\\hat{i}$": COLOR_X,
                "$\\hat{j}$": COLOR_Y,
                "Area": YELLOW,
                "Volume": YELLOW
            }

            for line in desc_lines:
                t = Tex(line, font_size=28, tex_to_color_map=kw_colors)
                if t.width > 5.0:
                    t.scale_to_fit_width(5.0)
                descriptions.add(t)
            
            descriptions.arrange(DOWN, aligned_edge=LEFT, buff=0.15)
            descriptions.next_to(matrix_mobj, DOWN, buff=0.5).align_to(title, LEFT)
            
            return VGroup(title, matrix_mobj, descriptions)

        # --- HELPER FOR ANIMATION SEQUENCE ---
        def run_transformation(title, matrix_vals, description_list, run_time=2):
            # 1. Show Matrix
            m_mob = Matrix(matrix_vals, h_buff=0.7)
            m_mob.get_entries()[0].set_color(COLOR_X) 
            m_mob.get_entries()[3].set_color(COLOR_Y) 
            # Highlight shear terms if present
            if matrix_vals[0][1] != 0: m_mob.get_entries()[1].set_color(COLOR_X)
            if matrix_vals[1][0] != 0: m_mob.get_entries()[2].set_color(COLOR_Y)
            
            m_group = VGroup(MathTex("M = "), m_mob).arrange(RIGHT)
            sidebar = create_sidebar(title, m_group, description_list)
            
            self.play(Write(sidebar))
            
            # 2. Transform Grid
            # The background_plane stays still (Reference). 
            # The grid_group transforms (Visualizing the matrix).
            self.play(
                grid_group.animate.apply_matrix(matrix_vals, about_point=origin_point),
                run_time=run_time
            )
            self.wait(2)
            
            # 3. Reset
            det = matrix_vals[0][0]*matrix_vals[1][1] - matrix_vals[0][1]*matrix_vals[1][0]
            if abs(det) > 0.0001:
                inv_mat = [
                    [matrix_vals[1][1]/det, -matrix_vals[0][1]/det],
                    [-matrix_vals[1][0]/det, matrix_vals[0][0]/det]
                ]
                self.play(
                    grid_group.animate.apply_matrix(inv_mat, about_point=origin_point),
                    FadeOut(sidebar),
                    run_time=1.0
                )
            else:
                self.play(FadeOut(sidebar))

        # --- ANIMATION START ---
        # 1. Draw static background first
        self.play(DrawBorderThenFill(background_plane), run_time=1)
        
        # 2. Fade in moving grid (it matches background initially)
        self.play(
            FadeIn(moving_plane),
            GrowArrow(i_hat), Write(i_label),
            GrowArrow(j_hat), Write(j_label),
            GrowArrow(vector), Write(vector_label)
        )
        self.wait(0.5)

        # --- SCALING ANIMATIONS ---
        
        # 1. Uniform Scaling
        run_transformation(
            "1. Uniform Scaling",
            [[2, 0], [0, 2]],
            [
                r"Scales $x$ and $y$ equally.",
                r"$\bullet$ Grid expands evenly",
                r"$\bullet$ $\vec{v}$ angle stays $45^\circ$",
                r"$\bullet$ Area grows by $2 \times 2 = 4$"
            ]
        )

        # 2. X-Axis Scaling
        run_transformation(
            "2. X-Direction Scaling",
            [[2, 0], [0, 1]],
            [
                r"Stretches only $x$ (Red lines).",
                r"$\bullet$ $\hat{i}$ doubles, $\hat{j}$ fixed",
                r"$\bullet$ $\vec{v}$ tilts down ($<45^\circ$)",
                r"$\bullet$ Area doubles (det=2)"
            ]
        )

        # 3. Y-Axis Scaling
        run_transformation(
            "3. Y-Direction Scaling",
            [[1, 0], [0, 2]],
            [
                r"Stretches only $y$ (Green lines).",
                r"$\bullet$ $\hat{j}$ doubles, $\hat{i}$ fixed",
                r"$\bullet$ $\vec{v}$ tilts up ($>45^\circ$)",
                r"$\bullet$ Area doubles (det=2)"
            ]
        )

        # 4. Non-Uniform Scaling
        run_transformation(
            "4. Non-Uniform Scaling",
            [[0.5, 0], [0, 1.5]],
            [
                r"Squish $x$, Stretch $y$.",
                r"$\bullet$ Red lines get closer",
                r"$\bullet$ Green lines get further",
                r"$\bullet$ Area scales by $0.75$"
            ]
        )

        # --- SHEARING ANIMATIONS ---

        # 5. Horizontal Shear (Positive)
        run_transformation(
            "5. Horizontal Shear (+)",
            [[1, 1], [0, 1]],
            [
                r"Push top to the right.",
                r"$\bullet$ $\hat{i}$ fixed",
                r"$\bullet$ $\hat{j}$ leans right",
                r"$\bullet$ Grid $\to$ Parallelograms",
                r"$\bullet$ Area stays constant (det=1)"
            ]
        )

        # 6. Vertical Shear (Positive)
        run_transformation(
            "6. Vertical Shear (+)",
            [[1, 0], [1, 1]],
            [
                r"Push right side up.",
                r"$\bullet$ $\hat{j}$ fixed",
                r"$\bullet$ $\hat{i}$ leans up",
                r"$\bullet$ Vertical lines tilt",
                r"$\bullet$ Area stays constant (det=1)"
            ]
        )

         # 7. Horizontal Shear (Negative)
        run_transformation(
            "7. Horizontal Shear (-)",
            [[1, -1], [0, 1]],
            [
                r"Push top to the left.",
                r"$\bullet$ $\hat{j}$ leans left",
                r"$\bullet$ $\vec{v}$ angle increases",
                r"$\bullet$ Volume/Area preserved"
            ]
        )

        # --- CONCLUSION ---
        conclusion = Text("Linear Transformations\nwarp space linearly.", font_size=36, color=YELLOW).to_edge(LEFT)
        self.play(Write(conclusion))
        self.wait(2)

%manim -ql -v warning ScalingAndShearing

Manim Community v0.19.0

/tmp/ipykernel_55009/104629532.py:26: DeprecationWarning: This method is not guaranteed to stay around. Please prefer setting the attribute normally or with Mobject.set().
  background_plane.set_width(7)
/tmp/ipykernel_55009/104629532.py:49: DeprecationWarning: This method is not guaranteed to stay around. Please prefer setting the attribute normally or with Mobject.set().
  moving_plane.set_width(7).move_to(origin_point)
